#  Day 5 — Decision Tree Classifier
## Heart Disease Prediction

Predicts whether a patient has heart disease using a Decision Tree trained on medical attributes.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

import warnings
warnings.filterwarnings('ignore')
%matplotlib inline

## 1. Load & Explore Dataset

In [ ]:
df = pd.read_csv('dataset/heart.csv')
print('Shape:', df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.isnull().sum()

## 2. Exploratory Data Analysis

In [ ]:
# Target Distribution
plt.figure(figsize=(6,4))
colors = ['#2ecc71', '#e74c3c']
counts = df['target'].value_counts().sort_index()
bars = plt.bar(['No Disease', 'Heart Disease'], counts.values, color=colors, edgecolor='white')
for bar, val in zip(bars, counts.values):
    plt.text(bar.get_x()+bar.get_width()/2, bar.get_height()+2, str(val),
             ha='center', va='bottom', fontsize=12, fontweight='bold')
plt.title('Target Distribution', fontsize=14, fontweight='bold')
plt.ylabel('Count')
plt.tight_layout()
plt.savefig('screenshots/target_distribution.png', dpi=150)
plt.show()

In [ ]:
# Correlation Heatmap
plt.figure(figsize=(12,8))
sns.heatmap(df.corr(), cmap='coolwarm', annot=True, fmt='.2f',
            linewidths=0.5, annot_kws={'size':7})
plt.title('Feature Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('screenshots/correlation_heatmap.png', dpi=150)
plt.show()

## 3. Prepare Features & Split Data

In [ ]:
X = df.drop('target', axis=1)
y = df['target']
print('Features:', list(X.columns))

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f'Train: {X_train.shape}  |  Test: {X_test.shape}')

## 4. Train Decision Tree (Gini vs Entropy)

In [ ]:
# Gini
model_gini = DecisionTreeClassifier(criterion='gini', max_depth=5, random_state=42)
model_gini.fit(X_train, y_train)
acc_gini = accuracy_score(y_test, model_gini.predict(X_test))

# Entropy
model_entropy = DecisionTreeClassifier(criterion='entropy', max_depth=5, random_state=42)
model_entropy.fit(X_train, y_train)
acc_entropy = accuracy_score(y_test, model_entropy.predict(X_test))

print(f'Gini    accuracy: {acc_gini:.4f}')
print(f'Entropy accuracy: {acc_entropy:.4f}')

# Use Gini as final model
model = model_gini

## 5. Evaluation

In [ ]:
pred = model.predict(X_test)
acc  = accuracy_score(y_test, pred)
print(f'Accuracy: {acc:.4f}\n')
print(classification_report(y_test, pred, target_names=['No Disease','Heart Disease']))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, pred)
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['No Disease','Heart Disease'],
            yticklabels=['No Disease','Heart Disease'])
plt.title(f'Confusion Matrix  |  Accuracy: {acc:.1%}', fontsize=13, fontweight='bold')
plt.ylabel('Actual'); plt.xlabel('Predicted')
plt.tight_layout()
plt.savefig('screenshots/confusion_matrix.png', dpi=150)
plt.show()

## 6. Feature Importance

In [ ]:
importance = model.feature_importances_
feat_df = pd.DataFrame({'Feature': X.columns, 'Importance': importance}).sort_values('Importance')

plt.figure(figsize=(8,6))
colors_bar = plt.cm.RdYlGn(np.linspace(0.2, 0.9, len(feat_df)))
plt.barh(feat_df['Feature'], feat_df['Importance'], color=colors_bar)
plt.title('Feature Importance — Decision Tree', fontsize=13, fontweight='bold')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.savefig('screenshots/feature_importance.png', dpi=150)
plt.show()

print('Top 3 features:')
print(feat_df.tail(3)[['Feature','Importance']].to_string(index=False))

## 7. Visualize Decision Tree

In [ ]:
plt.figure(figsize=(20, 10))
plot_tree(model,
          feature_names=list(X.columns),
          class_names=['No Disease', 'Heart Disease'],
          filled=True, rounded=True, fontsize=8)
plt.title('Decision Tree — Heart Disease Prediction', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('screenshots/decision_tree.png', dpi=120)
plt.show()

## 8. Save Model

In [ ]:
pickle.dump(model,            open('model.pkl',        'wb'))
pickle.dump(list(X.columns),  open('feature_names.pkl', 'wb'))
print(' model.pkl saved')
print(' feature_names.pkl saved')
print(f'\nFinal Accuracy: {acc:.2%}')